<a href="https://colab.research.google.com/github/amasha-sinhalagoda/J26-IT-334/blob/model%2Fcp_landslide/converttogeoJSON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, pandas as pd, requests, time, os, pickle
from google.colab import drive

# Mount Google Drive and set up the project folder
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/landslide_project"; os.makedirs(SAVE, exist_ok=True)

# Load metadata and prepare coordinate grids for SRTM elevation querying
st = pd.read_csv("static_clean_step1.csv")
URL = "https://api.opentopodata.org/v1/srtm30m"
STEP = 30.0; m_per_deg = 111320.0
lat = st["meta_lat"].to_numpy(); lon = st["meta_lon"].to_numpy()
dlat = STEP / m_per_deg; dlon = STEP / (m_per_deg * np.cos(np.radians(lat)))
offsets = [(+1,-1),(+1,0),(+1,+1),(0,-1),(0,0),(0,+1),(-1,-1),(-1,0),(-1,+1)]
coords = [(lat[k] + di*dlat, lon[k] + dj*dlon[k]) for k in range(len(st)) for di, dj in offsets]

# Fetch elevation data in batches using checkpointing to handle potential API interruptions
ck = f"{SAVE}/static_elev_checkpoint.pkl"
out = pickle.load(open(ck, "rb")) if os.path.exists(ck) else []
print("coordinates:", len(coords), "| already done:", len(out))
BATCH = 100
for i in range(len(out), len(coords), BATCH):
    chunk = coords[i:i+BATCH]; loc = "|".join(f"{la:.6f},{lo:.6f}" for la, lo in chunk)
    for attempt in range(5):
        r = requests.post(URL, json={"locations": loc, "interpolation": "bilinear"}, timeout=120)
        if r.status_code == 200:
            out.extend([x["elevation"] for x in r.json()["results"]]); break
        time.sleep(3 * (attempt + 1))
    else:
        raise RuntimeError(f"failed at {i}: {r.status_code}")
    if (i // BATCH) % 25 == 0: pickle.dump(out, open(ck, "wb")); print(len(out), "of", len(coords))
    time.sleep(1.1)
pickle.dump(out, open(ck, "wb"))

# Calculate elevation and terrain slope using finite-difference formulas, then export the data
elev = np.array(out, dtype="float64").reshape(len(st), 9)
a, b, c, d, e, f, g, h, i = [elev[:, j] for j in range(9)]
dz_dx = ((c + 2*f + i) - (a + 2*d + g)) / (8 * STEP)
dz_dy = ((a + 2*b + c) - (g + 2*h + i)) / (8 * STEP)
st["elevation_m"] = e; st["slope_deg"] = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy)))
st.to_csv(f"{SAVE}/static_with_terrain.csv", index=False)
print("NaN left:", int(st[["elevation_m", "slope_deg"]].isna().sum().sum()))

Mounted at /content/drive
coordinates: 48933 | already done: 0
100 of 48933
2600 of 48933
5100 of 48933
7600 of 48933
10100 of 48933
12600 of 48933
15100 of 48933
17600 of 48933
20100 of 48933
22600 of 48933
25100 of 48933
27600 of 48933
30100 of 48933
32600 of 48933
35100 of 48933
37600 of 48933
40100 of 48933
42600 of 48933
45100 of 48933
47600 of 48933
NaN left: 0


In [4]:
import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd
from google.colab import drive
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Initialize working directory, load temporal dataset and define feature groups
drive.mount("/content/drive")
SEED = 42
SAVE = "/content/drive/MyDrive/landslide_project"

path = f"{SAVE}/landslide_dynamic_v3_temporal.csv"
if not os.path.exists(path):
    from google.colab import files
    print("Not found in Drive. Choose landslide_dynamic_v3_temporal.csv")
    files.upload()
    path = "landslide_dynamic_v3_temporal.csv"

v3 = pd.read_csv(path, parse_dates=["event_date"])
y = v3["dynamic_label"].values
SOIL = [c for c in v3.columns if c.startswith("soil_")]
TERRAIN = ["elevation_m", "slope_deg"]
RAIN = ["rain_1d_mm","rain_3d_mm","rain_7d_mm","rain_14d_mm","rain_30d_mm","max_daily_rain_7d_mm","rainfall_days_7d"]
pos_w = (y == 0).sum() / (y == 1).sum()

# Helper function to generate standardized pipelines for the classifiers
def make_model(name):
    if name == "Logistic Regression":
        return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000, class_weight="balanced", random_state=SEED))
    if name == "XGBoost":
        return XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.04, subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                             reg_lambda=5, scale_pos_weight=pos_w, eval_metric="logloss", n_jobs=2, random_state=SEED)
    return LGBMClassifier(n_estimators=250, max_depth=3, num_leaves=8, learning_rate=0.04, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          min_child_samples=20, reg_lambda=5, class_weight="balanced", verbose=-1, n_jobs=2, random_state=SEED)

print(v3.shape, v3.row_type.value_counts().to_dict(), "| soil cols:", len(SOIL), "| missing rain:", int(v3[RAIN].isna().sum().sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Not found in Drive. Choose landslide_dynamic_v3_temporal.csv


Saving landslide_dynamic_v3_temporal.csv to landslide_dynamic_v3_temporal.csv
(1437, 32) {'event': 479, 'spatial_control': 479, 'temporal_control': 479} | soil cols: 10 | missing rain: 21


In [5]:
# Define feature combinations and establish rainfall test scenarios (ordinary vs. storm conditions)
feats = SOIL + TERRAIN + RAIN
scen = {"ordinary": v3.loc[v3.row_type == "temporal_control", RAIN].median(),
        "storm":    v3.loc[v3.row_type == "event", RAIN].median()}

SAVE = "/content/drive/MyDrive/landslide_project"
st = pd.read_csv(f"{SAVE}/static_with_terrain.csv")
for name in scen: st[f"score_{name}"] = np.nan

# Train models dynamically out-of-fold and predict risk scores across the scenarios
for k in sorted(v3["fold"].unique()):
    tr = (v3["fold"] != k).values
    model = make_model("LightGBM").fit(v3.loc[tr, feats], y[tr])
    te = (st["fold"] == k).values
    for name, rv in scen.items():
        X = st.loc[te, SOIL + TERRAIN].copy()
        for col in RAIN: X[col] = rv[col]
        st.loc[te, f"score_{name}"] = model.predict_proba(X[feats])[:, 1]

print(st[["score_ordinary", "score_storm"]].describe().round(2).loc[["count", "mean", "min", "max"]])

       score_ordinary  score_storm
count         5437.00      5437.00
mean             0.01         0.38
min              0.00         0.01
max              0.16         0.98


In [6]:
import json

# Group dynamic prediction scores into landslide risk tiers and export points as a GeoJSON collection
TIERS = ["Very Low", "Low", "Moderate", "High", "Very High"]
tier = lambda s: TIERS[int(np.digitize(s, [.2, .4, .6, .8]))]
st = st.dropna(subset=["score_storm", "score_ordinary"]).reset_index(drop=True)

features = []
for i, r in st.iterrows():
    features.append({"type": "Feature",
        "geometry": {"type": "Point", "coordinates": [round(float(r.meta_lon), 5), round(float(r.meta_lat), 5)]},
        "properties": {"id": f"LS-{i:04d}", "hazard": "landslide",
                       "score": round(float(r.score_storm), 2), "tier": tier(r.score_storm),
                       "score_ordinary": round(float(r.score_ordinary), 2), "tier_ordinary": tier(r.score_ordinary),
                       "date": "Scenario: cyclone-level rainfall", "demo": False}})
json.dump({"type": "FeatureCollection", "features": features}, open(f"{SAVE}/landslide_real.geojson", "w"))

from google.colab import files
files.download(f"{SAVE}/landslide_real.geojson")
print(len(features), "features exported")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

5437 features exported
